# 02 — Dynamic beta estimation

> Esta notebook compara una actualización gradual de intercepto y betas con el benchmark rolling OLS de 252 sesiones. Es una revisión causal y descriptiva sobre una muestra ya explorada.

## <b>benchmark congelado y construccion de Kalman </b>

El benchmark reproduce la atribución de `01_factor_stability_diagnostics`: datos Gold, MKT, SIZE y GV, intercepto, ventana rolling de 252 sesiones y betas conocidas hasta $t-1$.

El estado es $theta_t = [alpha_t, beta_{MKT,t}, beta_{SIZE,t}, beta_{GV,t}]$ y evoluciona como $theta_t = theta_{t-1} + eta_t$, con $eta_t \sim N(0, Q)$. La observación es $R_t = x_t theta_t + epsilon_t$, con $x_t = [1, MKT_t, SIZE_t, GV_t]$ y $epsilon_t \sim N(0, R)$.

> Para atribuir la sesión $t$, el filtro usa el estado predicho con información hasta $t-1$. Sólo después de registrar ese error observa $R_t$ y actualiza el estado para la sesión siguiente.

In [1]:
from datetime import date
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots
from IPython.display import HTML, display

TARGETS = ('QQQ', 'XLE', 'XLF')
SYMBOLS = ('SPY', 'QQQ', 'IWM', 'IWD', 'IWF', 'XLE', 'XLF')
FACTORS = ('market', 'size', 'growth_value')
RESEARCH_START = date(2018, 1, 2)
RESEARCH_END = date(2026, 11, 17)
CALIBRATION_END = date(2021, 12, 31)
EVALUATION_START = date(2022, 1, 1)
ROLLING_WINDOW = 252

def project_root(start):
    for parent in (start.resolve(), *start.resolve().parents):
        direct = parent / 'Momentum_systematic_strategies'
        if (direct / 'data' / 'gold' / 'daily_series' / 'etf').is_dir():
            return direct
        if (parent / 'data' / 'gold' / 'daily_series' / 'etf').is_dir():
            return parent
    raise FileNotFoundError('No se encontró data/gold/daily_series/etf.')

PROJECT_ROOT = project_root(Path.cwd())
GOLD = PROJECT_ROOT / 'data' / 'gold' / 'daily_series' / 'etf'
frames = {
    symbol: pl.read_parquet(GOLD / f'{symbol}.parquet')
    .with_columns(pl.col('session_date').cast(pl.Date))
    .filter(pl.col('session_date').is_between(RESEARCH_START, RESEARCH_END, closed='both'))
    .sort('session_date')
    for symbol in SYMBOLS
}
prices = frames['SPY'].select(pl.col('session_date').alias('date'), pl.col('adjusted_close').alias('SPY'))
for symbol in SYMBOLS[1:]:
    prices = prices.join(
        frames[symbol].select(pl.col('session_date').alias('date'), pl.col('adjusted_close').alias(symbol)),
        on='date', how='inner',
    )
returns = prices.sort('date').with_columns([pl.col(symbol).log().diff().alias(f'{symbol}_ret') for symbol in SYMBOLS]).drop_nulls()
model_panel = returns.select('date', *[f'{asset}_ret' for asset in TARGETS], 'SPY_ret', 'IWM_ret', 'IWF_ret', 'IWD_ret').with_columns(
    pl.col('SPY_ret').alias('market'),
    (pl.col('IWM_ret') - pl.col('SPY_ret')).alias('size'),
    (pl.col('IWF_ret') - pl.col('IWD_ret')).alias('growth_value'),
).select('date', *[f'{asset}_ret' for asset in TARGETS], *FACTORS).drop_nulls()

dates = model_panel.get_column('date').to_list()
factor_values = model_panel.select(*FACTORS).to_numpy()
evaluation_index = next(index for index, value in enumerate(dates) if value >= EVALUATION_START)
assert dates[evaluation_index - 1] <= CALIBRATION_END
model_panel.select('date').head(1), model_panel.select('date').tail(1)

(shape: (1, 1)
 ┌────────────┐
 │ date       │
 │ ---        │
 │ date       │
 ╞════════════╡
 │ 2018-01-03 │
 └────────────┘,
 shape: (1, 1)
 ┌────────────┐
 │ date       │
 │ ---        │
 │ date       │
 ╞════════════╡
 │ 2026-11-17 │
 └────────────┘)

## <b>Calibration before evaluation</b>

`R` se calibra como la varianza de los errores OOS de rolling OLS disponibles hasta 2021. `Q` es diagonal y toma la varianza de los cambios de un día en los coeficientes rolling de ese mismo periodo. El último OLS disponible al cierre de 2021 inicializa el estado, y su covarianza residual inicializa $P$. No se prueban configuraciones alternativas.

In [2]:
ols_rows, calibration = [], {}
for asset in TARGETS:
    actual = model_panel.get_column(f'{asset}_ret').to_numpy()
    coefficients_history, residual_history = [], []
    for index in range(ROLLING_WINDOW, len(actual)):
        design = np.column_stack((np.ones(ROLLING_WINDOW), factor_values[index - ROLLING_WINDOW:index]))
        beta = np.linalg.lstsq(design, actual[index - ROLLING_WINDOW:index], rcond=None)[0]
        predicted = np.r_[1.0, factor_values[index]] @ beta
        residual = actual[index] - predicted
        row = {'date': dates[index], 'beta_asof_date': dates[index - 1], 'asset': asset,
               'alpha': beta[0], 'beta_market': beta[1], 'beta_size': beta[2],
               'beta_growth_value': beta[3], 'actual_return': actual[index],
               'predicted_return': predicted, 'residual_return': residual}
        ols_rows.append(row)
        if dates[index] <= CALIBRATION_END:
            coefficients_history.append(beta)
            residual_history.append(residual)
    history = np.asarray(coefficients_history)
    q = np.diag(np.var(np.diff(history, axis=0), axis=0, ddof=1))
    r = float(np.var(residual_history, ddof=1))
    initial_beta = history[-1]
    last_design = np.column_stack((np.ones(ROLLING_WINDOW), factor_values[evaluation_index - ROLLING_WINDOW:evaluation_index]))
    initial_p = r * np.linalg.pinv(last_design.T @ last_design)
    calibration[asset] = {'Q': q, 'R': r, 'theta': initial_beta, 'P': initial_p}

ols = pl.DataFrame(ols_rows)
reference_r2 = {'QQQ': 0.972, 'XLE': 0.559, 'XLF': 0.865}
benchmark_replication_rows = []
for asset in TARGETS:
    sample = ols.filter(pl.col('asset') == asset)
    actual = sample.get_column('actual_return').to_numpy()
    error = sample.get_column('residual_return').to_numpy()
    r2 = 1 - (error ** 2).sum() / ((actual - actual.mean()) ** 2).sum()
    benchmark_replication_rows.append({'asset': asset, 'rolling_ols_r2': r2, 'reference_r2': reference_r2[asset],
                                      'abs_difference': abs(r2 - reference_r2[asset])})
benchmark_replication = pl.DataFrame(benchmark_replication_rows).with_columns(pl.all().exclude('asset').round(6))
if benchmark_replication.get_column('abs_difference').max() > 0.0005:
    raise AssertionError('El benchmark rolling OLS no reproduce la notebook anterior.')
calibration_summary = pl.DataFrame([
    {'asset': asset, 'R': item['R'], 'Q_alpha': item['Q'][0, 0], 'Q_MKT': item['Q'][1, 1],
     'Q_SIZE': item['Q'][2, 2], 'Q_GV': item['Q'][3, 3]}
    for asset, item in calibration.items()
]).with_columns(pl.all().exclude('asset').round(10))
benchmark_replication, calibration_summary

(shape: (3, 4)
 ┌───────┬────────────────┬──────────────┬────────────────┐
 │ asset ┆ rolling_ols_r2 ┆ reference_r2 ┆ abs_difference │
 │ ---   ┆ ---            ┆ ---          ┆ ---            │
 │ str   ┆ f64            ┆ f64          ┆ f64            │
 ╞═══════╪════════════════╪══════════════╪════════════════╡
 │ QQQ   ┆ 0.971996       ┆ 0.972        ┆ 0.000004       │
 │ XLE   ┆ 0.558685       ┆ 0.559        ┆ 0.000315       │
 │ XLF   ┆ 0.865383       ┆ 0.865        ┆ 0.000383       │
 └───────┴────────────────┴──────────────┴────────────────┘,
 shape: (3, 6)
 ┌───────┬──────────┬────────────┬──────────┬──────────┬──────────┐
 │ asset ┆ R        ┆ Q_alpha    ┆ Q_MKT    ┆ Q_SIZE   ┆ Q_GV     │
 │ ---   ┆ ---      ┆ ---        ┆ ---      ┆ ---      ┆ ---      │
 │ str   ┆ f64      ┆ f64        ┆ f64      ┆ f64      ┆ f64      │
 ╞═══════╪══════════╪════════════╪══════════╪══════════╪══════════╡
 │ QQQ   ┆ 0.000006 ┆ 2.0000e-10 ┆ 0.000005 ┆ 0.00001  ┆ 0.000017 │
 │ XLE   ┆ 0.000188 ┆

In [6]:
benchmark_replication

asset,rolling_ols_r2,reference_r2,abs_difference
str,f64,f64,f64
"""QQQ""",0.971996,0.972,0.000004
"""XLE""",0.558685,0.559,0.000315
"""XLF""",0.865383,0.865,0.000383


## <b>Causal Kalman attribution</b>

Para cada retorno de evaluación, primero se predice con $theta_{t|t-1}$, después se registra el error y finalmente se actualiza con el retorno observado. La actualización de $t$ no participa en la atribución de $t$.

In [3]:
kalman_rows = []
identity = np.eye(4)
for asset in TARGETS:
    actual = model_panel.get_column(f'{asset}_ret').to_numpy()
    theta = calibration[asset]['theta'].copy()
    covariance = calibration[asset]['P'].copy()
    q, r = calibration[asset]['Q'], calibration[asset]['R']
    for index in range(evaluation_index, len(actual)):
        x = np.r_[1.0, factor_values[index]]
        predicted_theta = theta
        predicted_covariance = covariance + q
        predicted_return = x @ predicted_theta
        residual = actual[index] - predicted_return
        innovation_variance = float(x @ predicted_covariance @ x + r)
        gain = predicted_covariance @ x / innovation_variance
        theta = predicted_theta + gain * residual
        update = identity - np.outer(gain, x)
        covariance = update @ predicted_covariance @ update.T + np.outer(gain, gain) * r
        kalman_rows.append({'date': dates[index], 'beta_asof_date': dates[index - 1], 'asset': asset,
                            'alpha': predicted_theta[0], 'beta_market': predicted_theta[1],
                            'beta_size': predicted_theta[2], 'beta_growth_value': predicted_theta[3],
                            'actual_return': actual[index], 'predicted_return': predicted_return,
                            'residual_return': residual})
kalman = pl.DataFrame(kalman_rows)
causality_audit = pl.DataFrame([{
    'ols_rows': ols.filter(pl.col('date') >= EVALUATION_START).height,
    'kalman_rows': kalman.height,
    'ols_betas_precede_return': (ols.filter(pl.col('date') >= EVALUATION_START).get_column('beta_asof_date') < ols.filter(pl.col('date') >= EVALUATION_START).get_column('date')).all(),
    'kalman_betas_precede_return': (kalman.get_column('beta_asof_date') < kalman.get_column('date')).all(),
}])
causality_audit

ols_rows,kalman_rows,ols_betas_precede_return,kalman_betas_precede_return
i64,i64,bool,bool
3672,3672,true,true


## <b>Attribution comparison</b>

Las métricas se calculan sobre las mismas fechas de 2022–2026. $R^2_{OOS} = 1 - SSE/SST$ y el percentil 95 resume errores absolutos extremos sin convertirlos en una prueba de trading.

In [4]:
ols_evaluation = ols.filter(pl.col('date') >= EVALUATION_START).select('date', 'asset', 'alpha', 'beta_market', 'beta_size', 'beta_growth_value', 'actual_return', 'predicted_return', 'residual_return')
comparison_rows = []
for model, frame in (('Rolling OLS 252d', ols_evaluation), ('Kalman', kalman)):
    for asset in TARGETS:
        sample = frame.filter(pl.col('asset') == asset).sort('date')
        actual = sample.get_column('actual_return').to_numpy()
        error = sample.get_column('residual_return').to_numpy()
        beta_values = sample.select('beta_market', 'beta_size', 'beta_growth_value').to_numpy()
        comparison_rows.append({
            'asset': asset, 'model': model, 'n': sample.height,
            'mae_pct': np.abs(error).mean() * 100, 'rmse_pct': np.sqrt(np.mean(error ** 2)) * 100,
            'oos_r2': 1 - (error ** 2).sum() / ((actual - actual.mean()) ** 2).sum(),
            'p95_abs_error_pct': np.quantile(np.abs(error), .95) * 100,
            'beta_std_mean': beta_values.std(axis=0, ddof=1).mean(),
            'beta_mean_abs_daily_change': np.abs(np.diff(beta_values, axis=0)).mean(),
        })
comparison_summary = pl.DataFrame(comparison_rows).with_columns(pl.all().exclude('asset', 'model', 'n').round(4))
comparison_summary

subperiod_rows = []
for model, frame in (('Rolling OLS 252d', ols_evaluation), ('Kalman', kalman)):
    for asset in TARGETS:
        for label, start, end in (('2022', date(2022, 1, 1), date(2022, 12, 31)), ('2023–2026', date(2023, 1, 1), RESEARCH_END)):
            sample = frame.filter((pl.col('asset') == asset) & pl.col('date').is_between(start, end, closed='both'))
            error = sample.get_column('residual_return').to_numpy()
            subperiod_rows.append({'asset': asset, 'model': model, 'period': label, 'n': sample.height,
                                   'mae_pct': np.abs(error).mean() * 100, 'rmse_pct': np.sqrt(np.mean(error ** 2)) * 100,
                                   'p95_abs_error_pct': np.quantile(np.abs(error), .95) * 100})
subperiod_errors = pl.DataFrame(subperiod_rows).with_columns(pl.all().exclude('asset', 'model', 'period', 'n').round(4))
subperiod_errors

asset,model,period,n,mae_pct,rmse_pct,p95_abs_error_pct
str,str,str,i64,f64,f64,f64
"""QQQ""","""Rolling OLS 252d""","""2022""",251,0.1685,0.2176,0.4426
"""QQQ""","""Rolling OLS 252d""","""2023–2026""",973,0.1874,0.2715,0.5417
"""XLE""","""Rolling OLS 252d""","""2022""",251,1.4233,1.7729,3.441
"""XLE""","""Rolling OLS 252d""","""2023–2026""",973,0.8961,1.1989,2.3658
"""XLF""","""Rolling OLS 252d""","""2022""",251,0.4447,0.5737,1.1137
…,…,…,…,…,…,…
"""QQQ""","""Kalman""","""2023–2026""",973,0.1803,0.2623,0.5062
"""XLE""","""Kalman""","""2022""",251,1.4271,1.7767,3.5391
"""XLE""","""Kalman""","""2023–2026""",973,0.8824,1.1715,2.3408


## <b>XLE adaptation and stability</b>

> Estas trayectorias permiten ver rapidez de adaptación y ruido de actualización. Ningún trazo, por sí solo, identifica una beta económicamente correcta.

In [7]:
xle_ols = ols_evaluation.filter(pl.col('asset') == 'XLE').sort('date')
xle_kalman = kalman.filter(pl.col('asset') == 'XLE').sort('date')
figure = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=('MKT beta', 'SIZE beta', 'GV beta'))
for row, column in enumerate(('beta_market', 'beta_size', 'beta_growth_value'), start=1):
    figure.add_scatter(x=xle_ols['date'], y=xle_ols[column], mode='lines', name='Rolling OLS' if row == 1 else None,
                       line={'color': '#636EFA'}, legendgroup='ols', showlegend=row == 1, row=row, col=1)
    figure.add_scatter(x=xle_kalman['date'], y=xle_kalman[column], mode='lines', name='Kalman' if row == 1 else None,
                       line={'color': '#EF553B'}, legendgroup='kalman', showlegend=row == 1, row=row, col=1)
figure.update_layout(title='XLE — predicted factor exposures before each return', template='plotly_white', height=700)
display(HTML(figure.to_html(include_plotlyjs='cdn', full_html=False)))


In [8]:
error_rows = []
for model, frame in (('Rolling OLS 252d', xle_ols), ('Kalman', xle_kalman)):
    error = np.abs(frame.get_column('residual_return').to_numpy()) * 100
    for index in range(62, len(error)):
        error_rows.append({'date': frame.get_column('date')[index], 'model': model,
                           'rolling_p95_abs_error_pct': np.quantile(error[index - 62:index + 1], .95)})
rolling_errors = pl.DataFrame(error_rows)
figure = go.Figure()
for model in ('Rolling OLS 252d', 'Kalman'):
    sample = rolling_errors.filter(pl.col('model') == model)
    figure.add_scatter(x=sample['date'], y=sample['rolling_p95_abs_error_pct'], mode='lines', name=model)
figure.update_layout(title='XLE — 63-session rolling p95 absolute attribution error', xaxis_title='return date', yaxis_title='error (%)', template='plotly_white')
display(HTML(figure.to_html(include_plotlyjs='cdn', full_html=False)))

## <b>Conclusion</b>

- Kalman mejora modestamente las métricas agregadas de los tres activos. En XLE, MAE pasa de 1.0042% a 0.9941%, $R^2$ de 0.3164 a 0.3351 y p95 de 2.7145% a 2.6863%.
- La mejora de XLE se concentra en 2023–2026; en 2022 su MAE y p95 son ligeramente peores que rolling OLS. No hay evidencia de superioridad uniforme por subperiodo.
- La dispersión de betas de XLE es casi igual bajo ambos métodos (0.2890 frente a 0.2869), aunque Kalman actualiza algo más cada día (0.0102 frente a 0.0081). La adaptación no aparece explosiva bajo esta calibración.
- QQQ y XLF también mejoran levemente, pero las magnitudes son pequeñas. La consistencia descriptiva no basta para seleccionar Kalman como modelo definitivo.
- $Q$ y $R$ se fijan con información anterior a 2022; quedan sin identificar la especificación óptima, los cambios discretos, los factores omitidos y cualquier capacidad predictiva o de trading.